# 08 Go 语言基础与并发

> 覆盖范围：Go 的设计理念（编译型/静态类型/GC）、基础语法（变量/控制流/函数/复合类型）、**切片与 map 底层**、struct 与接口（鸭子类型）、**goroutine 与 channel（CSP 模型）**、并发原语（WaitGroup/Mutex/select）、标准库常用包、Go 与 Python/C++ 对比、面试高频（内存逃逸/GMP/chan 原理）。
> 使用方式：先读**原理讲解**（为什么 Go 适合后端/云原生、并发模型为何如此设计），再看 **Go 代码示例**（默写），最后用 Python 模拟核心语义验证，自测面试追问。
> 注：本 notebook 内核为 Python 3；Go 片段用于阅读与默写，对照实验用 Python 模拟以保持可执行验证。

---

## 1. Go 是什么 / 为什么火

Go（2009，Google）是**静态类型、编译型、带 GC、原生并发**的语言，解决「C++ 编译慢/难写并发」与「Python 性能差」之间的工程矛盾：

- **编译为单个静态二进制**：部署 = 拷一个文件，无需运行时依赖（Docker/云原生绝配）；
- **编译快**：工程级项目秒级编译（对比 C++）；
- **原生并发**：goroutine + channel（CSP），比线程轻量得多；
- **大厂背书**：Kubernetes、Docker、etcd、Prometheus、influxdb 全是 Go；
- 缺点：GC 有停顿（不如 Rust 可控）、泛型 1.18 才引入、错误处理冗长（`if err != nil`）。

## 2. 基础语法速览（对照 Python/C++）

```go
package main

import "fmt"

func add(a, b int) int {          // 类型在后！返回值在后
    return a + b
}

func main() {
    // 变量：var 或 :=（类型推断）
    var x int = 1
    y := 2
    s := "hello"
    fmt.Println(add(x, y), s)

    // 控制流：if 不需要括号，for 是唯一的循环
    if y > 1 {
        fmt.Println("y > 1")
    }
    for i := 0; i < 3; i++ {
        fmt.Println(i)
    }
    // switch 默认不穿透（无需 break）
    switch y {
    case 1: fmt.Println("one")
    case 2: fmt.Println("two")
    }
}
```

**Go vs Python vs C++（面试常问选型）**：

| 维度 | Go | Python | C++ |
| --- | --- | --- | --- |
| 类型 | 静态 | 动态 | 静态 |
| 编译/运行 | 编译（单二进制） | 解释 | 编译（快） |
| 并发 | goroutine 原生 | GIL 受限 | 线程（复杂） |
| 内存 | GC | GC | 手动/RAII |
| 适用 | 后端/云原生/中间件 | AI/脚本/快速原型 | 系统/性能敏感 |

## 3. 切片（slice）与 map（面试高频）

### 3.1 slice = 视图三元组

`slice` 不是数组，而是 **{指针, 长度 len, 容量 cap}** 三字段的结构：

```go
s := make([]int, 3, 5)   // len=3 cap=5
s = append(s, 1)         // 不超 cap 就地；超 cap 申请新底层数组（翻倍）
t := s[1:3]              // 切片共享底层数组！改 t 会改 s（易错）
```

> ⚠️ **易错点**：多个 slice 共享底层数组，`append` 超过容量会**换新底层数组**导致「看似改了共享数组实际没改」——切片陷阱题常考。

### 3.2 map = 哈希表

```go
m := make(map[string]int)
m["a"] = 1
v, ok := m["b"]        // ok=false 表示键不存在（读不存在的键返回零值）
delete(m, "a")
```

- map 是**引用类型**，函数内修改外部可见；
- **多 goroutine 并发读写 map 会 panic**（不是数据竞争警告而是直接崩）——需要加锁或 sync.Map；
- 迭代顺序**不保证**（随机）——不要依赖 map 遍历顺序。

## 4. struct 与接口（鸭子类型）

```go
type Animal interface {        // 接口：只声明方法集合
    Speak() string
}

type Dog struct{ Name string }  // struct：字段

func (d Dog) Speak() string {   // 方法：接收者 receiver（值 or 指针）
    return "woof"
}

func zoo(a Animal) string { return a.Speak() }
// Dog 实现了 Speak() 就自动满足 Animal 接口——无需显式声明（鸭子类型）
```

**值接收者 vs 指针接收者**：值接收者拷贝副本（方法内修改不影响原对象）；指针接收者 `func (d *Dog)` 修改作用于原对象；结构体大时用指针避免拷贝。

**interface{} / any（空接口）**：可承载任何类型（像 dynamic/any），通过**类型断言** `v.(int)` 或 `switch v.(type)` 还原。

## 5. goroutine：比线程轻量得多

**goroutine = Go 的并发单元**（`go f()` 一行启动）：

- 初始栈 **2KB**（线程默认 1MB+），几百万 goroutine 没问题；
- 由 **GMP 调度器**把 goroutine 调度到少量 OS 线程上：
  - **G**（goroutine）、**M**（machine=OS 线程）、**P**（processor=本地队列）；
  - 一个 P 一个本地运行队列，M 从 P 取 G；阻塞/系统调用时 M 解绑、P 找新 M；
- 对比线程：goroutine 用户态调度、切换成本微秒级。

```go
func main() {
    for i := 0; i < 5; i++ {
        go func(n int) { fmt.Println(n) }(i)   // 注意：参数拷贝，不能直接用循环变量
    }
    time.Sleep(time.Second)  // 主函数返回程序就退出，需等待
}
```

## 6. channel：goroutine 间通信（CSP）

**Do not communicate by sharing memory; instead, share memory by communicating.**（不要通过共享内存通信，要通过通信共享内存）

```go
ch := make(chan int)       // 无缓冲：发送/接收必须配对（同步）
ch2 := make(chan int, 3)   // 有缓冲：缓冲区未满不阻塞发送

ch <- 1       // 发送
v := <-ch     // 接收
close(ch)     // 关闭：接收方不再阻塞（v, ok := <-ch 的 ok=false）

// select：多路等待（像 switch 但用于 channel）
select {
case v := <-ch:
    fmt.Println(v)
case <-time.After(time.Second):
    fmt.Println("timeout")   // 超时兜底
default:
    // 所有 case 都不可用时的非阻塞分支
}
```

**有缓冲 vs 无缓冲**：无缓冲 = 同步握手（发送方等接收方）；有缓冲 = 有限异步队列（生产者消费者经典模型）。**谁发送谁关闭**（关闭的 channel 再发送会 panic）。

## 7. 并发原语

```go
// WaitGroup：等待一组 goroutine
var wg sync.WaitGroup
for i := 0; i < 5; i++ {
    wg.Add(1)
    go func(n int) { defer wg.Done(); work(n) }(i)
}
wg.Wait()   // 阻塞直到 Done 计数归零

// Mutex：互斥锁保护共享资源
var mu sync.Mutex
mu.Lock()
count++
mu.Unlock()   // 通常 defer mu.Unlock()

// 生产者-消费者经典模式：chan + goroutine
jobs := make(chan int, 10)
go func() { for i := 0; i < 10; i++ { jobs <- i }; close(jobs) }()
for j := range jobs { process(j) }
```

## 8. 常用标准库

- `fmt`（I/O）、`strings`/`strconv`、`sort`、`time`、`os`/`io`、`encoding/json`、`net/http`、`context`（超时/取消传递）、`sync`。

```go
// 极简 HTTP 服务（5 行）
import "net/http"
http.HandleFunc("/", func(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintln(w, "Hello, Go")
})
http.ListenAndServe(":8080", nil)

## 9. GMP 调度细节（面试纵深）

goroutine 能「开几百万个」靠的是运行时调度器。先记三个字母：

- **G（Goroutine）**：一个待运行的任务（含栈、状态、PC）；
- **M（Machine）**：操作系统线程，真正干活的执行者；
- **P（Processor）**：本地运行队列 + 调度上下文，默认数量 = `GOMAXPROCS` = CPU 核数（可用 `runtime.GOMAXPROCS(n)` 改）。

```
M1 → P1 → [G1,G3,G5]   M2 → P2 → [G2]    ...  P 数 = CPU 核数
      ↑ work stealing：P 的本地队列空了，去别的 P 偷一半 G
全局队列：M 在本地队列和全局队列都没 G 时去取
```

**调度事件**：

- G 主动让出（channel 阻塞/`runtime.Gosched()`）→ 暂停，P 跑下一个 G；
- G 做系统调用（读文件/syscall）阻塞 → **M 与 P 解绑**，P 找别的 M 继续跑（保证 P 不闲置）；syscall 返回后 M 再排队；
- G 结束 → P 取新 G。

> **面试一句话**：GMP 让「用户态调度」替代「内核线程调度」——创建/切换成本低（2KB 栈、微秒级），所以百万 goroutine 可行；对比线程 1MB 栈、内核态切换。

## 10. 内存模型：逃逸分析与 GC

**逃逸分析（Escape Analysis）**：编译器决定变量**分配在栈还是堆**——

- 变量被**函数返回**、被**闭包捕获**、地址被**存进堆对象**→ 逃逸到堆（编译器在函数返回后仍能用）；
- 纯局部变量不逃逸 → 栈上分配（随函数返回自动回收，零 GC 开销）；
- 面试可答：返回 `&T{}` 是**合法且安全**的（Go 自动把 `T` 放到堆），不像 C/C++ 悬垂指针。

**垃圾回收（GC）**：

- 三色标记（白/灰/黑）+ 写屏障，**并发标记、小停顿**（STW 极短，毫秒级）；
- 垃圾回收触发：堆增长、`runtime.GC()`、内存压力；
- 对比：Java/C# 也有 GC，Go 的 GC 延迟目标低（几个 ms），代价是吞吐略低。

## 11. defer / panic / recover（陷阱集中区）

```go
func f() {
    defer fmt.Println("A")   // 延后到函数返回前执行
    defer fmt.Println("B")   // defer 是栈：后进先出（LIFO）→ 先打 B 再打 A
    fmt.Println("main")
}
```

- **defer LIFO**：多个 defer 逆序执行（像栈），常配合成对打开/关闭资源；
- **defer 参数立即求值**：`defer fmt.Println(x)` 的 x 在 defer 语句处就拷贝了；要用最终值需闭包 `defer func(){ fmt.Println(x) }()`；
- **defer + 命名返回值**：`defer func(){ r = 0 }()` 可以**修改返回值**（经典面试题）；
- **panic / recover**：`panic` 抛错，`recover` 只能在 **defer** 中调用才能截获（模拟 try-catch），之后函数返回但程序不崩；
- ⚠️ 不要在主 goroutine 里 recover 别的 goroutine 的 panic——**recover 只能救当前 goroutine**（每个 goroutine 独立）。

## 12. 并发模式实战

```go
// ① worker pool：固定数量 worker 消费任务（限流/资源控制）
jobs := make(chan int, 100)
var wg sync.WaitGroup
for w := 0; w < 4; w++ {                 // 4 个 worker
    wg.Add(1)
    go func() {
        defer wg.Done()
        for j := range jobs { process(j) }   // 从 channel 取任务
    }()
}
for j := 0; j < 100; j++ { jobs <- j }
close(jobs)                              // 关管道，worker 取完退出
wg.Wait()

// ② context：超时/取消传播（HTTP、RPC 必用）
ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
defer cancel()
// select { case r := <-ch: ; case <-ctx.Done(): return ctx.Err() }  // 超时取消

// ③ errgroup：一组 goroutine 任一失败即取消其余（golang.org/x/sync）
g, ctx := errgroup.WithContext(ctx)
for _, url := range urls { url := url; g.Go(func() error { return fetch(ctx, url) }) }
if err := g.Wait(); err != nil { log.Fatal(err) }
```

**goroutine 泄漏**（高频追问）：启动了却不退出（阻塞在无接收方的 channel/`select`），goroutine 和栈永不回收——用 `context` 取消或加超时兜底。


## 核心概念

### (a) 🗂️ Go：从「slice 底层」到「goroutine 并发」的主线

Go 的两大考点是**数据结构底层**（slice/map 怎么存）和**并发模型**（goroutine + channel 怎么通信）。把「slice 三要素 → goroutine 调度 → channel 通信 → 同步原语」串起来，并发题就有底。

```mermaid
flowchart TB
    A["🧠 slice 底层<br/>ptr / len / cap"]
    A --> B["🚀 goroutine<br/>轻量线程 / go 关键字"]
    B --> C["🔗 channel<br/>通信即同步 / 生产者-消费者"]
    C --> D["🛡️ 同步原语<br/>WaitGroup / Mutex"]
    D --> E["🎯 面试 30 秒速答<br/>陷阱与对比"]
```

- **slice 底层**：slice = 指针 + 长度 + 容量（ptr/len/cap），多个 slice 可能共享底层数组——`append` 扩容、切片陷阱（改一个全变）是必考题。
- **goroutine**：`go func()` 起一个轻量协程，由运行时调度到线程池上，不是操作系统线程，所以能起成千上万个。
- **channel**：`make(chan T)` 建管道，`ch <- v` 发送、`<-ch` 接收；未缓冲 channel 的收发会阻塞等待对方——「通信即同步」，生产者-消费者模型用它天然解耦。
- **同步原语**：WaitGroup 等一组 goroutine 干完，Mutex 保护共享数据；Go 官方推荐「不要通过共享内存通信，而要通过通信共享内存」。
- **面试 30 秒速答**：能说清 slice 陷阱、channel 阻塞语义、goroutine 与线程的差别，并发题就稳了。

## 本节要解决的问题

| 要解决的问题 | 对应内容 |
|---|---|
| 切片改了值为什么别的变量也变？ | slice 共享底层数组 |
| append 什么时候悄悄重新分配？ | 扩容策略 len/cap |
| map 遍历为什么顺序不稳定？ | 哈希桶与随机起始 |
| 上万 goroutine 为什么不卡？ | GMP 调度模型 |
| channel 为什么「通信即同步」？ | 阻塞与缓冲语义 |
| 并发计数怎么保证不出错？ | WaitGroup / Mutex / 原子 |

## 故事引入

你们组用 Go 写网关，压测时发现一个诡异 bug：`append` 之后，历史请求的数据「莫名其妙」被改成了新值。查了半天——原来是**切片共享底层数组**：`b := a[:3]` 没有拷贝数据，b 和 a 指着同一块内存，谁改谁都互相影响。

另一个场景：服务要同时处理 1 万个 WebSocket 连接，每个连接一个 goroutine，居然没爆内存也没崩——因为 goroutine 是**协程**（几 KB 栈），由 Go 运行时调度到少量 OS 线程上，而不是一连接一线程。

这两件事就是 Go 的灵魂：**数据结构要懂底层（slice/map）**，**并发要用原语（goroutine/channel）**。这篇把 Go 的底层与并发机制一次讲清，你就能写出既快又不踩坑的并发程序。

## 深入原理

### ① slice 三要素与共享底层数组

**直觉**：slice 不是数组，而是数组的「视图」：一个结构体 `{ptr, len, cap}`——ptr 指向底层数组起点，len 是当前长度，cap 是容量上限。切片 `b := a[1:3]` 只是改了 ptr/len，**没有复制数据**，所以 a、b 共享底层数组，改谁都互相可见。

**要点**：易错点：`b := a[:2]; b[0] = 99` 会把 a[0] 也改成 99；向 b append 若没超 cap 也会写到底层数组，可能「污染」a 的相邻位置。

**面试怎么问**：「两个 slice 共享底层数组时，什么操作会互相影响？」——改元素（同下标）和 append（未超 cap 时）都会；重新分配后（超 cap）就分道扬镳。

![slice 结构、共享数组与扩容流程](images/go_slice.svg)

> 📌 slice 共享底层数组：b := a[1:3] 不拷贝，改 b 连带改 a，append 超 cap 才换数组

### ② append 扩容：len/cap 怎么涨

**直觉**：`append(s, x)` 若 len < cap，直接写入并 len++（不拷贝）；若 len == cap，**重新分配**更大底层数组（cap 增长策略：<256 翻倍，>=256 约 1.25 倍），拷贝旧数据，再写入。所以 append 的返回值**必须重新赋值**：`s = append(s, x)`。

**要点**：`make([]int, 0, 100)` 预分配可避免反复扩容拷贝；`copy(dst, src)` 才是真正拷贝数据。

**面试怎么问**：「append 为什么必须接返回值？」——因为可能触发扩容换底层数组，原变量还指着旧数组。

### ③ map 底层：bucket 与哈希

**直觉**：Go 的 map 是**哈希表**：key 经哈希函数落到 bucket（桶），桶里存 8 个键值对，满了用溢出桶链起来。因为 key 会重新哈希且 bucket 随机分布，所以**遍历顺序不稳定**——不能依赖 map 的顺序。

**要点**：读取不存在的 key 返回零值（`v, ok := m[k]` 用 ok 判断存在）；map 是引用类型，`nil` map 只能读不能写（写会 panic）。

**面试怎么问**：「为什么 map 遍历顺序每次都不同？」——运行时在遍历起点随机化 bucket 起始位置，防依赖顺序的代码。

### ④ goroutine 与 GMP 调度

**直觉**：goroutine 是 Go 运行时的**协程**：栈初始几 KB、动态伸缩，创建成本远低于 OS 线程。GMP 调度器把海量 goroutine（G）放到少量 OS 线程（M）上，配合逻辑处理器（P）并行执行——所以「一万个 goroutine」完全可行。

**要点**：`go func()` 启动；main 退出程序即退出（不等待 goroutine）；`runtime.GOMAXPROCS` 默认 = CPU 核数。

**面试怎么问**：「goroutine 和线程的区别？」——栈小可伸缩、用户态调度切换快、数量可达百万级；线程栈 1-8MB、内核调度。

![GMP 调度与 work stealing](images/go_gmp.svg)

> 📌 GMP 调度：G(协程)→P(处理器本地队列)→M(OS 线程)→CPU，用户态切换开销极小

### ⑤ channel 阻塞与 select

**直觉**：channel 是 goroutine 之间的「数据管道」：`ch <- v` 发送、`v := <-ch` 接收。**无缓冲 channel** 的发送会阻塞直到对方接收——这就叫「通信即同步」（发送和接收同时发生）；**有缓冲 channel**（`make(chan T, n)`）在缓冲区满前不阻塞。

**要点**：`select` 可同时监听多个 channel，谁有数据走谁；`close(ch)` 通知接收方结束（`v, ok := <-ch` 里 ok=false）。

**面试怎么问**：「无缓冲 channel 和有缓冲的本质区别？」——前者是同步握手（数据不落地），后者是队列（生产消费解耦，可异步）。

![channel 缓冲、阻塞与关闭流程](images/go_channel.svg)

> 📌 无缓冲 channel 发送等接收（通信即同步）；有缓冲 channel 是队列，满了才阻塞

### ⑥ 同步原语：WaitGroup / Mutex / 原子

**直觉**：**WaitGroup** 等一组 goroutine 干完（`Add(1)` / `Done()` / `Wait()`）；**Mutex** 锁住临界区（共享变量读写要加锁）；**原子操作**（`sync/atomic`）对单个 int 变量做无锁 CAS/加减。Go 的哲学：**不要通过共享内存通信，要通过通信共享内存**——能用 channel 就不用锁。

**要点**：`var wg sync.WaitGroup; for ... { wg.Add(1); go func(){ defer wg.Done(); ... }() }; wg.Wait()`；`atomic.AddInt64(&n, 1)` 比 Mutex 更轻。

**面试怎么问**：「什么时候用 Mutex 什么时候用 channel？」——共享可变状态用 Mutex；任务流水线/通知用 channel；单个计数器用 atomic。

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 模拟 slice 底层（ptr/len/cap）与共享底层数组陷阱

class Slice:
    def __init__(self, array, start, end, cap=None):
        self.array = array          # 底层数组引用（共享！）
        self.start = start
        self.len = end - start
        self.cap = cap if cap is not None else len(array) - start

    def __getitem__(self, i):
        if i < 0 or i >= self.len:
            raise IndexError
        return self.array[self.start + i]

    def __setitem__(self, i, v):
        self.array[self.start + i] = v

    def append(self, v):
        if self.len < self.cap:
            self.array[self.start + self.len] = v
            self.len += 1                     # 未超 cap：就地写共享数组
            return self
        # 超 cap：申请新底层数组（模拟翻倍），原数组不再共享
        new = list(range(len(self.array)))    # 模拟扩容
        new = self.array[:] + [None] * max(len(self.array), 1)
        new[self.start + self.len] = v
        self.array = new
        self.len += 1
        self.cap = len(new) - self.start
        return self

    def __repr__(self):
        return f'Slice(len={self.len},cap={self.cap},data={self.array[self.start:self.start+self.len]})'

# 共享底层数组：s 和 t 同源
base = list(range(10))
s = Slice(base, 0, 5, cap=10)     # s = base[0:5]  cap=10
t = Slice(base, 2, 4)             # t = base[2:4]（共享同一个底层数组）
print('初始:', s, '|', t)
t[0] = 99                          # 改 t 影响 s（共享底层）
print('改 t[0]=99 后 s:', s, ' <- 共享数组被修改')
assert s[2] == 99

# append 不超 cap：仍然共享；超 cap：换新数组
s.append(5)                        # len 5->6，cap 10 够，仍共享
print('append 后 s:', s)
print('<- Go 中切片陷阱：append 后另一个共享数组的 slice 可能“看不到这个元素”')

arr = [1, 2, 3, 4, 5]
u = Slice(arr, 0, 2, cap=2)        # len=2 cap=2
u.append(100)                      # 超 cap -> 新数组
print('超 cap append:', u, ' | 原数组:', arr, ' <- 原数组不受影响')
assert arr == [1, 2, 3, 4, 5]

In [ ]:
# 模拟 goroutine + channel 生产者-消费者（关注不需要锁的通信模型）

import threading, queue, time, random

# Python 的 queue.Queue 相当 Go 的有缓冲 channel（简化演示）
ch = queue.Queue(maxsize=3)
results = []

def producer():
    for i in range(5):
        ch.put(i)                     # 类似 ch <- i（缓冲满时阻塞）
        print(f'[producer] 发送 {i}')
        time.sleep(0.03)
    
def consumer():
    while True:
        try:
            v = ch.get(timeout=0.3)   # 类似 v := <-ch
            results.append(v)
            print(f'[consumer] 收到 {v}')
        except queue.Empty:
            print('[consumer] 通道关闭/超时，退出')
            break

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start(); t2.start()
t1.join(); t2.join()
print('\n最终收到的数据:', sorted(results))
assert sorted(results) == list(range(5)), '生产者消费者通过 channel 应无丢失无重复'
print('<- Go 中 goroutine + channel 实现同样的“通信而非共享锁”模式')

# select 超时模拟
def select_timeout(chans, timeout_s):
    """模拟 select{case v:=<-ch: ... case <-time.After(t): timeout}"""
    done = threading.Event()
    def waiter():
        done.wait(timeout_s)
    w = threading.Thread(target=waiter); w.start()
    # 简并：主线程等数据或超时
    return 'timeout' if done.wait(timeout_s) is False or not results else 'got-data'

print('select 自动超时:', select_timeout([], 0.05))

In [ ]:
# 模拟 WaitGroup 与 Mutex：理解并发编排原语

import threading, time

class WaitGroup:
    """模拟 sync.WaitGroup：Add / Done / Wait"""
    def __init__(self):
        self.cv = threading.Condition()
        self.count = 0
    def add(self, n=1):
        with self.cv:
            self.count += n
    def done(self):
        with self.cv:
            self.count -= 1
            if self.count <= 0:
                self.cv.notify_all()
    def wait(self):
        with self.cv:
            self.cv.wait_for(lambda: self.count <= 0)

wg = WaitGroup()
done_jobs = []

def work(i):
    time.sleep(0.02)
    done_jobs.append(i)
    wg.done()                         # defer wg.Done()

for i in range(5):
    wg.add(1)                         # wg.Add(1)
    threading.Thread(target=work, args=(i,)).start()
wg.wait()                             # wg.Wait()：阻塞到全部完成
print('WaitGroup 等待全部完成, 已完成任务:', sorted(done_jobs))
assert sorted(done_jobs) == list(range(5))

# Mutex 模拟：并发计数（无锁会丢更新）
count = {'v': 0}
lock = threading.Lock()

def inc_no_lock():
    for _ in range(10000):
        count['v'] += 1      # 非原子

threads = [threading.Thread(target=inc_no_lock) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f'无锁并发计数: {count["v"]} (期望 80000，会丢更少)')

count2 = {'v': 0}
def inc_lock():
    for _ in range(10000):
        with lock:
            count2['v'] += 1

threads = [threading.Thread(target=inc_lock) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f'加锁并发计数: {count2["v"]} (期望 80000)')
assert count2['v'] == 80000, 'Go 中 sync.Mutex 保证互斥，计数应精确'
print('<- 对应 Go: sync.Mutex 保护共享计数，避免数据竞争')

## 13. 面试 30 秒速答

- **Go 特点**：静态类型、编译型单二进制、GC、原生并发；K8s/Docker 生态；
- **goroutine vs 线程**：goroutine 2KB 栈、GMP 用户态调度、百万级；线程 1MB 栈、内核调度；
- **channel（CSP）**：通信而非共享内存；有缓冲=有限异步队列、无缓冲=同步握手；谁发送谁 close；
- **GMP**：G=goroutine、M=OS 线程、P=本地队列；阻塞时 M 解绑 P 换新 M，保证 P 不闲置；
- **slice 陷阱**：底层数组共享；append 超 cap 换新数组；多 slice 同源改一处影响另一处；
- **map 并发**：多 goroutine 并发读写 map 直接 panic；用 sync.Mutex 或 sync.Map；
- **接口**：鸭子类型，结构体实现方法即满足接口；值 vs 指针接收者；
- **错误处理**：多返回值 `(value, err)`，`if err != nil` 显式检查（不像异常）。
- **逃逸分析**：返回指针/闭包捕获 → 变量放堆；纯局部 → 栈；GC 三色标记、停顿毫秒级；
- **defer**：LIFO 栈序；参数立即求值、闭包延迟；recover 只能救当前 goroutine；
- **并发模式**：worker pool（固定 worker + channel）、context 超时取消、errgroup 任一失败取消全部；
- **泄漏**：goroutine 阻塞在无人接收的 channel 上不退出即泄漏——用 context/超时兜底。

## 14. 自测清单（面试追问）

- [ ] Go 对比 Python/C++ 的优劣？什么场景选 Go？（后端/云原生/中间件）
- [ ] goroutine 与线程有什么区别？为什么能开几百万个？GMP 里 P 是什么角色？
- [ ] channel 有缓冲与无缓冲的区别？close 后发送会怎样？谁负责 close？
- [ ] 手写：生产者消费者（channel）；WaitGroup 等 5 个 goroutine；select 超时。
- [ ] slice 的 len/cap 关系？append 超 cap 时底层数组会怎样？共享数组改一处为何影响另一处？
- [ ] map 并发读写为什么会 panic？（并发写节点/扩容）怎么解决？
- [ ] 值接收者与指针接收者区别？什么时候用指针？
- [ ] 空接口 interface{} 是什么？类型断言怎么写？switch v.(type) 呢？
- [ ] Go 的错误处理为什么用返回值而不是异常？
- [ ] defer 的执行顺序？（后进先出）；defer + 修改返回值 的经典题
- [ ] 手写：一个极简 HTTP 服务器；一个带 context.WithTimeout 的请求。
- [ ] GMP 中 M 阻塞（syscall）时会发生什么？为什么要 P 与 M 解绑？（保证 P 不闲置）
- [ ] 什么是逃逸分析？哪些情况会把变量分配到堆上？
- [ ] defer 是 LIFO 吗？`defer fmt.Println(x)` 与 `defer func(){fmt.Println(x)}()` 区别？
- [ ] recover 为什么必须在 defer 里？能 recover 其他 goroutine 的 panic 吗？
- [ ] 手写：worker pool（4 个 worker 处理 100 个任务）；context 超时取消。
- [ ] goroutine 泄漏是怎么产生的？怎么避免？

> ✅ 全部能回答 + 能默写代码，本课通过。

## 面试考点（Q&A）

**Q1：b := a[1:3]，改 b[0] 会影响 a 吗？**
A：会——b 与 a 共享底层数组，b[0] 就是 a[1]。💡 追问：怎么断开共享？——`copy` 到新 slice。

**Q2：append 什么时候会重新分配内存？**
A：len == cap 时；扩容后旧 slice 与新 slice 不再共享底层数组。💡 追问：扩容策略？——<256 翻倍，>=256 约 1.25 倍。

**Q3：为什么 map 遍历顺序不稳定？**
A：哈希表 bucket 分布 + 运行时随机起点。💡 追问：要稳定顺序怎么办？——取 key 排序后遍历。

**Q4：goroutine 和 OS 线程的区别？**
A：goroutine 栈小（KB 级）、用户态调度、百万级；线程栈大、内核调度、数量受限。💡 追问：GMP 是什么？——G(协程)M(线程)P(处理器) 三层调度模型。

**Q5：无缓冲 channel 的发送会怎样？**
A：阻塞直到有接收方，同时握手。💡 追问：怎么避免死锁？——确保收发配对，或用 select+default 非阻塞。

**Q6：并发计数 1 万次，怎么保证结果正确？**
A：`atomic.AddInt64` 或 Mutex 包裹，或 channel 串行。💡 追问：普通 n++ 为什么错？——读-改-写非原子，多个 goroutine 互相覆盖。

## 小结与下一步

Go 的并发三件套：**goroutine**（便宜的执行单元）+ **channel**（通信即同步）+ **同步原语**（锁/原子兜底）。加上 slice/map 的底层认知（共享数组、哈希桶），你已经能写出正确又高效的并发程序。

**下一步**：进入 09-Java 与 JVM——从「怎么并发」切换到「内存怎么管、垃圾怎么回收」，理解 JVM 的运行时数据区与 GC。